In [2]:
# 1. Install Nilearn (This fixes the ModuleNotFoundError)
!pip install nilearn

import os
from google.colab import drive
from nilearn import datasets

# 2. Mount Drive
drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/ASD_GNN_Research2'

# 3. Ensure your exact folder structure exists
subdirs = [
    'results/metrics', 'results/figures',
    'raw_data', 'preprocessed/time_series',
    'notebooks', 'graphs/train', 'graphs/val', 'graphs/test',
    'models/GCN', 'models/GAT', 'models/Checkpoints',
    'connectivity/metrics'
]
for d in subdirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# 4. Fetch ABIDE directly into raw_data
print("Downloading ABIDE dataset to your Google Drive...")
abide_data = datasets.fetch_abide_pcp(
    data_dir=os.path.join(base_dir, 'raw_data'),
    pipeline='cpac',
    derivatives=['rois_aal'],
    quality_checked=True
)
print("Raw data downloaded and structured successfully!")

Mounted at /content/drive


[fetch_abide_pcp] Dataset directory found: /content/drive/MyDrive/ASD_GNN_Research2/raw_data/ABIDE_pcp

Raw data downloaded and structured successfully!


Time-Series Extraction and Connectivity Calculation

In [ ]:
import numpy as np
import pandas as pd
import os
from nilearn.connectome import ConnectivityMeasure

# 1. Define Paths based on your Drive structure
base_dir = '/content/drive/MyDrive/ASD_GNN_Research2'
raw_data_path = os.path.join(base_dir, 'raw_data')
ts_dir = os.path.join(base_dir, 'preprocessed', 'time_series')
conn_dir = os.path.join(base_dir, 'connectivity')

# 2. Extract and Clean Phenotypic Data
# The DX_GROUP column contains the labels: 1 for Autism, 2 for Typical Control.
# We map this to 1 (ASD) and 0 (Control) for standard binary classification.
pheno_df = pd.DataFrame(abide_data.phenotypic)
pheno_df['DX_GROUP'] = pheno_df['DX_GROUP'].map({1: 1, 2: 0})

print("Extracting time-series arrays and standardizing shapes...")
valid_subjects = []
time_series_list = []

# 3. Process Time-Series Data
# The AAL atlas standardizes the brain into exactly 116 regions.
for idx, sub_id in enumerate(pheno_df['SUB_ID']):
    ts_data = abide_data.rois_aal[idx]

    # Nilearn may return file paths (strings to .1D files) or direct numpy arrays
    if isinstance(ts_data, str):
        ts_array = np.loadtxt(ts_data)
    else:
        ts_array = ts_data

    # Strict Quality Control: Ensure the matrix has exactly 116 regions
    if len(ts_array.shape) > 1 and ts_array.shape[1] == 116:
        np.save(os.path.join(ts_dir, f'sub_{sub_id}_ts.npy'), ts_array)
        valid_subjects.append(sub_id)
        time_series_list.append(ts_array)
    else:
        print(f"Skipping Subject {sub_id} due to shape mismatch.")

print(f"Successfully saved clean time-series for {len(valid_subjects)} subjects.")

# 4. Filter and Save the Cleaned Labels CSV
pheno_clean = pheno_df[pheno_df['SUB_ID'].isin(valid_subjects)]
pheno_clean.to_csv(os.path.join(raw_data_path, 'phenotypic_cleaned.csv'), index=False)

# 5. Compute Functional Connectivity Matrices
print("\nComputing Functional Connectivity Matrices (Pearson Correlation)...")
conn_measure = ConnectivityMeasure(kind='correlation')

# fit_transform calculates the r-values between all 116 regions across all time points
connectivity_matrices = conn_measure.fit_transform(time_series_list)

# Save the computed graphs to your connectivity folder
for idx, sub_id in enumerate(valid_subjects):
    np.save(os.path.join(conn_dir, f'sub_{sub_id}_conn.npy'), connectivity_matrices[idx])

print("Connectivity calculation complete. Matrices saved to Drive!")

Extracting time-series arrays and standardizing shapes...
Successfully saved clean time-series for 871 subjects.

Computing Functional Connectivity Matrices (Pearson Correlation)...
Connectivity calculation complete. Matrices saved to Drive!


Graph Construction & Dataset Splitting

In [ ]:
# 1. Install PyTorch Geometric and its dependent wheels for Colab
!pip install torch_geometric
!pip install optional_dependencies torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.0+cu121.html

import os
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split

# 2. Update paths to use your folder name
base_dir = '/content/drive/MyDrive/ASD_GNN_Research2'
raw_data_path = os.path.join(base_dir, 'raw_data')
conn_dir = os.path.join(base_dir, 'connectivity')
graphs_base_dir = os.path.join(base_dir, 'graphs')

# Load the cleaned phenotypic data to get labels and subject IDs
pheno_clean = pd.read_csv(os.path.join(raw_data_path, 'phenotypic_cleaned.csv'))

# 3. Define your Graph Construction Parameters
CORRELATION_THRESHOLD = 0.5
graph_list = []

print("Converting connectivity matrices into PyTorch Geometric graph objects...")

for idx, row in pheno_clean.iterrows():
    subject_id = row['SUB_ID']
    label = row['DX_GROUP'] # 1 = ASD, 0 = Control

    matrix_path = os.path.join(conn_dir, f'sub_{subject_id}_conn.npy')

    # Double-check file existence to prevent missing index errors
    if not os.path.exists(matrix_path):
        continue

    matrix = np.load(matrix_path)

    # Extract edge coordinates where correlation exceeds the threshold (excluding self-loops)
    edge_indices = np.where((matrix > CORRELATION_THRESHOLD) & (~np.eye(matrix.shape[0], dtype=bool)))

    # Convert to PyTorch long tensors for PyG adjacency lists [2, num_edges]
    edge_index = torch.tensor(np.array(edge_indices), dtype=torch.long)

    # Extract correlation values to serve as edge attributes/weights
    edge_attr = torch.tensor(matrix[edge_indices], dtype=torch.float).unsqueeze(1)

    # Node features: One-hot identity matrix mapping for the 116 AAL brain regions
    num_nodes = matrix.shape[0]
    x = torch.eye(num_nodes, dtype=torch.float)

    # Classification label tensor
    y = torch.tensor([label], dtype=torch.long)

    # Construct the PyG Data Object
    graph_obj = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    graph_list.append((subject_id, graph_obj))

print(f"Generated {len(graph_list)} total graph structures.")

# 4. Stratified Splitting (Ensures balanced ASD/Control ratios across splits)
labels = [g[1].y.item() for g in graph_list]
train_val, test_data = train_test_split(graph_list, test_size=0.15, stratify=labels, random_state=42)

train_labels = [g[1].y.item() for g in train_val]
train_data, val_data = train_test_split(train_val, test_size=0.1765, stratify=train_labels, random_state=42)

# 5. Serialize and save the PyG .pt files to your target directories
splits = {
    'train': train_data,
    'val': val_data,
    'test': test_data
}

for split_name, dataset in splits.items():
    split_path = os.path.join(graphs_base_dir, split_name)
    print(f"Saving {len(dataset)} graphs to graphs/{split_name}...")

    for subject_id, data_obj in dataset:
        torch.save(data_obj, os.path.join(split_path, f'sub_{subject_id}.pt'))

print("\nGraph engineering pipeline complete! Data splits are fully serialized to Drive.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 970.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.2 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.3.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 90.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.6/949.6 kB 49.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing


Converting connectivity matrices into PyTorch Geometric graph objects...
Generated 871 total graph structures.
Saving 609 graphs to graphs/train...
Saving 131 graphs to graphs/val...
Saving 131 graphs to graphs/test...

Graph engineering pipeline complete! Data splits are fully serialized to Drive.


Implementing the GNN Models & Training Loop

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.loader import DataLoader
from torch_geometric.data import Dataset
from torch_geometric.nn import GATConv, global_mean_pool

# 1. Update Directory Configurations
base_dir = '/content/drive/MyDrive/ASD_GNN_Research2'
graphs_dir = os.path.join(base_dir, 'graphs')
checkpoint_dir = os.path.join(base_dir, 'models', 'Checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

# 2. Custom PyTorch Geometric Dataset Loader (UPDATED FIX)
class ABIDEGraphDataset(Dataset):
    def __init__(self, folder_path):
        super().__init__()
        self.folder_path = folder_path
        self.file_names = sorted([f for f in os.listdir(folder_path) if f.endswith('.pt')])

    def len(self):
        return len(self.file_names)

    def get(self, idx):
        file_path = os.path.join(self.folder_path, self.file_names[idx])
        # FIX: Explicitly set weights_only=False to bypass PyTorch 2.6 security restriction
        return torch.load(file_path, weights_only=False)

# Initialize split datasets
train_dataset = ABIDEGraphDataset(os.path.join(graphs_dir, 'train'))
val_dataset = ABIDEGraphDataset(os.path.join(graphs_dir, 'val'))
test_dataset = ABIDEGraphDataset(os.path.join(graphs_dir, 'test'))

# Instantiate DataLoaders (batching individual patient graphs together)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"DataLoaders Ready. Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

# 3. Model Architecture Options
class GATClassifier(torch.nn.Module):
    """
    Graph Attention Network (GAT) for Brain Network Classification.
    Provides built-in explainability via attention heads.
    """
    def __init__(self, num_node_features, hidden_channels, num_classes, heads=4):
        super(GATClassifier, self).__init__()

        # First GAT layer (Multi-head attention)
        self.conv1 = GATConv(num_node_features, hidden_channels, heads=heads, concat=True)
        # Second GAT layer (Aggregating heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False)

        # Dense classification head
        self.lin1 = Linear(hidden_channels, hidden_channels)
        self.lin2 = Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch, edge_attr=None):
        # 1. Node feature transformations via spatial attention
        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x = self.conv2(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)

        # 2. Global Pooling: Collapses node embeddings into a single graph vector
        x = global_mean_pool(x, batch)  # Shape: [batch_size, hidden_channels]

        # 3. Fully Connected Classifier
        x = self.lin1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.4, training=self.training)
        x = self.lin2(x)

        return x

# 4. Define Training and Validation Framework
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using execution device: {device}")

# Using 116 node features (matching our identity matrix dimensions)
model = GATClassifier(num_node_features=116, hidden_channels=64, num_classes=2, heads=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()
    total_loss = 0
    correct = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch, data.edge_attr.squeeze(-1) if data.edge_attr is not None else None)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
    return total_loss / len(train_dataset), correct / len(train_dataset)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total_loss = 0
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.batch, data.edge_attr.squeeze(-1) if data.edge_attr is not None else None)
        loss = criterion(out, data.y)
        total_loss += loss.item() * data.num_graphs
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# 5. Run the Training Loop
best_val_acc = 0.0
epochs = 50

print("\nBeginning Model Optimization Pipeline...")
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train()
    val_loss, val_acc = evaluate(val_loader)

    # Save checkpoint if validation metric improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint_path = os.path.join(checkpoint_dir, 'best_gat_model.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, checkpoint_path)
        print(f"Epoch {epoch:02d}: New Best Val Acc: {val_acc:.4f} -> Checkpoint Saved.")
    else:
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print(f"\nTraining Complete! Top Validation Accuracy: {best_val_acc:.4f}")

DataLoaders Ready. Train Batches: 20 | Val Batches: 5
Using execution device: cpu

Beginning Model Optimization Pipeline...
Epoch 01: New Best Val Acc: 0.5344 -> Checkpoint Saved.
Epoch 05: Train Loss: 0.6901, Train Acc: 0.5386 | Val Loss: 0.6910, Val Acc: 0.5344
Epoch 10: Train Loss: 0.6901, Train Acc: 0.5386 | Val Loss: 0.6908, Val Acc: 0.5344
Epoch 15: Train Loss: 0.6890, Train Acc: 0.5386 | Val Loss: 0.6911, Val Acc: 0.5344
Epoch 20: Train Loss: 0.6898, Train Acc: 0.5369 | Val Loss: 0.6912, Val Acc: 0.5344
Epoch 25: Train Loss: 0.6933, Train Acc: 0.5402 | Val Loss: 0.6909, Val Acc: 0.5344
Epoch 30: Train Loss: 0.6911, Train Acc: 0.5287 | Val Loss: 0.6914, Val Acc: 0.5344
Epoch 35: Train Loss: 0.6906, Train Acc: 0.5304 | Val Loss: 0.6913, Val Acc: 0.5344
Epoch 40: Train Loss: 0.6893, Train Acc: 0.5386 | Val Loss: 0.6909, Val Acc: 0.5344
Epoch 45: Train Loss: 0.6900, Train Acc: 0.5386 | Val Loss: 0.6909, Val Acc: 0.5344
Epoch 50: Train Loss: 0.6911, Train Acc: 0.5386 | Val Loss: 0.69